## Fremont Bridge Cycling Dashboard
### Visualisation Techniques 
**Author:** Arkadiusz Jedrzejewski 
**Date:** June 2026

In [7]:
# Install required packages
import sys
!{sys.executable} -m pip install dash plotly # APPENDING THIS LINE TO INSTALL DASH AND PLOTLY

# Create a Dash application
from dash import Dash, dcc, html, Input, Output
import pandas as pd
import numpy as np
import plotly.express as px

In [8]:
# Load the dataset
df = pd.read_csv('../DATASETS/Fremont_Bridge_-_Combined_Bicycle_and_Scooter_Counter_20260615.csv')

In [9]:
# Display the first 5 rows of the dataset
df.head(5)

,Date,"Fremont Bridge Sidewalks, south of N 34th St Total","Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk","Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk"
0,05/31/2026 11:00:00 PM,33,9.0,24.0
1,05/31/2026 10:00:00 PM,51,20.0,31.0
2,05/31/2026 09:00:00 PM,64,30.0,34.0
3,05/31/2026 08:00:00 PM,148,72.0,76.0
4,05/31/2026 07:00:00 PM,184,85.0,99.0


In [10]:
# Rename columns for easier access
df.columns = ['Date', 'Total', 'West_Sidewalk', 'East_Sidewalk']
df.head(5)

,Date,Total,West_Sidewalk,East_Sidewalk
0,05/31/2026 11:00:00 PM,33,9.0,24.0
1,05/31/2026 10:00:00 PM,51,20.0,31.0
2,05/31/2026 09:00:00 PM,64,30.0,34.0
3,05/31/2026 08:00:00 PM,148,72.0,76.0
4,05/31/2026 07:00:00 PM,184,85.0,99.0


In [11]:
# Basic dataset information
df.shape
df.info()
df.describe() 
df.isnull().sum()  # Check for missing values  


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119747 entries, 0 to 119746
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           119747 non-null  object 
 1   Total          119717 non-null  object 
 2   West_Sidewalk  119717 non-null  float64
 3   East_Sidewalk  119717 non-null  float64
dtypes: float64(2), object(2)
memory usage: 3.7+ MB


Date              0
Total            30
West_Sidewalk    30
East_Sidewalk    30
dtype: int64

In [13]:
# Convert 'Date' to datetime and 'Total' to numeric
df['Date'] = pd.to_datetime(df['Date'])

# Remove rows with missing values and duplicates
df['Total'] = df['Total'].astype(str)
df['Total'] = df['Total'].str.replace(',', '')
df['Total'] = pd.to_numeric(df['Total'], errors='coerce')

# Remove rows with missing values and duplicates
df.dropna(inplace=True)          
df.drop_duplicates(inplace=True)


# Total is already numeric, no conversion needed
df['Total'] = df['Total'].astype(int)

df.info()



<class 'pandas.core.frame.DataFrame'>
Index: 119717 entries, 0 to 119746
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Date           119717 non-null  datetime64[ns]
 1   Total          119717 non-null  int64         
 2   West_Sidewalk  119717 non-null  float64       
 3   East_Sidewalk  119717 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1)
memory usage: 4.6 MB


In [14]:
# Convert 'West_Sidewalk' and 'East_Sidewalk' to numeric
df['West_Sidewalk'] = df['West_Sidewalk'].astype(int)
df['East_Sidewalk'] = df['East_Sidewalk'].astype(int)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 119717 entries, 0 to 119746
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Date           119717 non-null  datetime64[ns]
 1   Total          119717 non-null  int64         
 2   West_Sidewalk  119717 non-null  int64         
 3   East_Sidewalk  119717 non-null  int64         
dtypes: datetime64[ns](1), int64(3)
memory usage: 4.6 MB


In [15]:
# extracting time features from Date column
df['Hour']      = df['Date'].dt.hour
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Month']     = df['Date'].dt.month
df['Year']      = df['Date'].dt.year

df.head(5)

,Date,Total,West_Sidewalk,East_Sidewalk,Hour,DayOfWeek,Month,Year
0,2026-05-31 23:00:00,33,9,24,23,Sunday,5,2026
1,2026-05-31 22:00:00,51,20,31,22,Sunday,5,2026
2,2026-05-31 21:00:00,64,30,34,21,Sunday,5,2026
3,2026-05-31 20:00:00,148,72,76,20,Sunday,5,2026
4,2026-05-31 19:00:00,184,85,99,19,Sunday,5,2026


In [10]:
# Create a Dash application
app = Dash()

# Define the layout of the dashboard, Source: McQuaid (2026) Dash Tutorial structure
app.layout = html.Div([

    html.H1("Fremont Bridge Cycling Dashboard"),
    html.Hr(),
    html.P("Data from Seattle Department of Transportation"),

    html.Label("Direction:"),
    dcc.RadioItems(
        id="direction",
        options=['Total', 'West_Sidewalk', 'East_Sidewalk'],
        value='Total',
        inline=True
    ),

    html.Label("Hours:"),
    dcc.RangeSlider(
        id="hour-slider",
        min=0, max=23, step=1,
        marks={0:"0:00", 6:"6:00", 12:"12:00", 18:"18:00", 23:"23:00"},
        value=[6, 20],
        tooltip={"always_visible": True, "placement": "bottom"}
    ),

    html.Hr(),
    dcc.Graph(id="graph")
])

# Update the graph based on user input
@app.callback(
    Output("graph", "figure"),
    Input("hour-slider", "value"),
    Input("direction", "value")
)
def update_chart(hour_range, direction):
    low, high = hour_range
    mask = (df['Hour'] >= low) & (df['Hour'] <= high)
    fig = px.line(df[mask], x='Date', y=direction,
                  title=f'Cyclists per Hour - {direction}')
    return fig

# Run the Dash app
app.run(debug=True, jupyter_mode='inline')

In [16]:
# Install additional required packages (dash-bootstrap-components and dash-bootstrap-templates)source: McQuaid (2026c) Styling Dash
import sys
!{sys.executable} -m pip install dash-bootstrap-components dash-bootstrap-templates

In [ ]:
# Import necessary libraries for styling, source: McQuaid (2026c) Styling Dash
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc
from dash_bootstrap_templates import load_figure_template

# Load the FLATLY template for styling, source: McQuaid (2026c) Styling Dash
load_figure_template('FLATLY')
app = Dash(external_stylesheets=[dbc.themes.FLATLY])



# Define the layout of the dashboard, source: McQuaid (2026) Dash Tutorial structure
app.layout = html.Div([

    html.H1("Fremont Bridge Cycling Dashboard",
            style={'fontFamily': 'Georgia, serif', 'color': '#1A2B4A', 'fontWeight': 'bold'} # Add font family, color, and weight for better visualization
            ),
    html.Hr(style={"borderColor": "#1A2B4A"}),  # Optional: Add border color for better visualization
    html.P("Data from Seattle Department of Transportation. ",
           style={'fontFamily': 'Arial', 'color': '#444444', 'fontSize': '14px', 'marginBottom': '2px'} # Add font family, color, and size for better visualization
           ),
    html.P("Source: https://data.seattle.gov/Transportation/Fremont-Bridge-Combined-Bicycle-and-Scooter-Counte/65db-xm6k/about_data",
           style={'fontFamily': 'Arial', 'color': '#888888', 'fontSize': '11px', 'marginTop': '1px'}), # Add font family, color, and size for better visualization

    html.Label("Direction:",
               style={'fontFamily': 'Arial', 'color': '#1A2B4A', 'fontWeight': 'bold'}
               ), # Add font family, color, and weight for better visualization
    dcc.RadioItems(
        id="direction",
        options=['Total', 'West_Sidewalk', 'East_Sidewalk'],
        value='Total',
        inline=True ,
        inputStyle={'accentColor': '#1B83D9'} # Add accent color for better visualization
        
    ),

    html.Label("Hours:",
               style={'fontFamily': 'Arial', 'color': '#1A2B4A', 'fontWeight': 'bold'}
               ),
    dcc.RangeSlider(
        id="hour-slider",
        min=0, max=23, step=1,
        marks={0:"0:00", 6:"6:00 a.m.", 12:"12:00 p.m.", 18:"6:00 p.m.", 23:"11:00 p.m."}, # Add custom marks for better visualization
        value=[6, 20],
        tooltip={"always_visible": True, "placement": "bottom"}
      
    ),

    html.Hr(
        style={'borderColor': '#1B83D9'}
    ), # Optional: Add border color for better visualization
    dcc.Graph(id="graph") 
],
style={"backgroundColor": "#E8EDF2", "padding": "20px"} # Add background and font color for better visualization
)

# Update the graph based on user input
@app.callback(
    Output("graph", "figure"),
    Input("hour-slider", "value"),
    Input("direction", "value")
)
def update_chart(hour_range, direction):
    low, high = hour_range
    mask = (df['Hour'] >= low) & (df['Hour'] <= high)
    fig = px.line(df[mask], x='Date', y=direction,
                  title=f'Cyclists per Hour - {direction}',
                  color_discrete_sequence=["#1B83D9"])  # Optional: Add blue color sequence for better visualization
    fig.update_layout(
        plot_bgcolor='#E8EDF2',
        paper_bgcolor="#BBD1E6",
        font=dict(color='#1A2B4A'),
        title_font=dict(size=18, color='#1565C0'),
        yaxis_title="Number of Cyclists", # Add y-axis title for better visualization
        xaxis_title="Actual Date", # Add x-axis title for better visualization
        yaxis=dict(gridcolor='#BBD1E6'), 
        xaxis=dict(gridcolor='#BBD1E6')

    )  # Optional: Set background and font color
    return fig

# Run the Dash app
app.run(debug=True, jupyter_mode='inline')

NameError: name 'Dash' is not defined